# 01. 온통청년 정책 데이터 전처리

온통청년 API 정책 데이터를 이용하여 전처리·분류·요약을 수행한 결과.

입력 파일은 `youth_policies_categorized.csv` 이고. 해당 파일과 같은 디렉토리에 위치해야 함.

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

candidate_paths = [
    BASE_DIR / "youth_policies_categorized.csv",
    BASE_DIR / "data" / "youth_policies_categorized.csv",
    BASE_DIR.parent / "youth_policies_categorized.csv",
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

print("현재 작업 폴더:", BASE_DIR)
print("입력 파일:", DATA_PATH)
if DATA_PATH is None:
    raise FileNotFoundError("youth_policies_categorized.csv를 찾지 못했습니다. 노트북과 같은 폴더에 파일을 두세요.")

try:
    df_raw = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
except UnicodeDecodeError:
    df_raw = pd.read_csv(DATA_PATH, encoding="cp949")

print("원본 크기:", df_raw.shape)
display(df_raw.head(3))
print(df_raw.columns.tolist())

현재 작업 폴더: C:\Users\yong\Desktop\TM\tm2\textmining_solo_policy_project\textmining_solo_policy_project
입력 파일: C:\Users\yong\Desktop\TM\tm2\textmining_solo_policy_project\textmining_solo_policy_project\youth_policies_categorized.csv
원본 크기: (5470, 27)


,지역,조회_zipCd,정책ID,정책명,정책키워드,정책설명,정책지원내용,정책대분류,정책중분류,자동분류,대표분류,신청기간,사업기간,지원대상,나이조건,소득조건,신청방법,제출서류,주관기관,운영기관,신청URL,참고URL1,참고URL2,최초등록일시,최종수정일시,수집페이지,수집출처
0,서울,11000,20260605005400113228,청년미래적금,보조금,"청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월 최대 50만원 한도 내에서 자유롭게 납입 가능(2026년 6월 22일 출시 예정)","은행이자+비과세 혜택+정부기여금(납입금액에 비례해 일반형 6%, 우대형 12%의 정부기여금 지원)",금융･복지･문화,취약계층 및 금융지원,복지,복지,20260622 ~ 20261231,20260622 ~ 20261231,"19세~34세 / 연령제한:N 0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상 0011009 0013010 0049010 0055003",19세~34세 / 연령제한:N,"0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상",ㅇ 취급은행 모바일앱을 통해 매월 비대면 신청 가능 ㅇ 2026년 6월 출시 예정,NaN,금융위원회,한국고용정보원,NaN,https://www.kinfa.or.kr/financialProduct/youthFutureSavings.do,https://blog.naver.com/blogfsc/224302863400,2026-06-05 18:06:49,2026-06-10 14:05:31,1,온통청년_OPEN_API_getPlcy
1,서울,11000,20260528005400113227,(농식품부) 농식품 바우처,바우처,『농업·농촌 및 식품산업 기본법』 제 23조의 2(취약계층 등에 대한 식품지원)에 근거하여 취약계층의 식품 접근성을 강화하고 균형 있는 식품 섭취를 지원하는 식품지원 제도,"- 지원방식: 전자바우처(카드방식) - 지원품목: 국산 채소류, 과일류, 육류, 신선알류, 흰 우유, 잡곡류, 두부류, 임산물 (이외 품목 구매 불가)",금융･복지･문화,건강,복지,복지,20251222 ~ 20261211,20260102 ~ 20261231,18세~34세 / 연령제한:N 0043003 0 0 생계급여(기준 중위소득 32%이하) 수급가구 중 임산부·영유아·아동·청년 포함가구 생계급여 수급가구 가구원 중 「국민기초생활 보장법」상 보장시설 수급자는 가구원 수 산출에서 제외 보건복지부 영...,18세~34세 / 연령제한:N,0043003 0 0 생계급여(기준 중위소득 32%이하) 수급가구 중 임산부·영유아·아동·청년 포함가구,1. 방문신청 : 주소지 관할 읍ㆍ면ㆍ동 행정복지센터 방문 2. 전화신청 : 고객지원센터 (1551-0857)를 통해 신청 3. 온라인신청 : 농식품 바우처 홈페이지에서 신청 4. 자동신청 : 2025년 농식품바우처 이용가구 중 2025. 12...,NaN,농림축산식품부,한국고용정보원,https://www.foodvoucher.go.kr/security/joinAgree,https://www.foodvoucher.go.kr/view/fm/vucintro/agriFood,NaN,2026-05-28 10:00:50,2026-05-28 10:01:12,1,온통청년_OPEN_API_getPlcy
2,서울,11000,20260528005400113226,(문체부) 청년예술인 예술활동 적립계좌,보조금,청년예술인에게 중장기 자산형성의 기회를 마련하여 안정적인 예술활동을 지원하는 사업,매월 일정 금액을 24개월간 적금 저축 시 가입자가 저축한 금액만큼 정부지원금을 지원 (1인 최대 240만원) - 상품종류: 10만원 정액 적금 - 가입기간: 2년 (24개월) - 납입한도: 월 10만원 (2년 최대 240만원) - 지급방식: ...,금융･복지･문화,예술인지원,복지,복지,NaN,20260101 ~ 20261231,"18세~39세 / 연령제한:N 0043002 0 3692 「예술인복지법」상 예술활동증명을 완료한 예술인(신청일 기준 예술활동증명 유효자) - 일반 예술활동증명 완료자(공개 발표된 예술활동, 예술활동 수입, 경력단절예술인, 특수한 작업방식, 무형...",18세~39세 / 연령제한:N,0043002 0 3692,1. 예술활동증명확인 예술인경력정보시스템(https://www.kawfartist.kr)을 접속하시어 경력지원 > 예술활동증명 > 신청내역 진행 상태에서 신청일 현재 예술활동증명 유효 여부를 확인합니다. 2. 구비서류 확인 상품 및 구비서류를 ...,"1. 주민등록초본 - 2026년 발급분 - 발급 시 발급대상자 본인 및 전체 발급 2. 소득금액증명원 - 2026년 발급분 - 귀속년도: 직전년도(2024년도) 소득금액증명원 발급 - 작성기준: 종합소득세 신고 현황의 소득금액(사업, 근로, 기...",문화체육관광부,한국고용정보원,https://www.artloan.kr/notice/savingsAccountProcess.do,https://www.artloan.kr/notice/savingsAccount.do,NaN,2026-05-28 09:34:16,2026-05-28 09:34:49,1,온통청년_OPEN_API_getPlcy


['지역', '조회_zipCd', '정책ID', '정책명', '정책키워드', '정책설명', '정책지원내용', '정책대분류', '정책중분류', '자동분류', '대표분류', '신청기간', '사업기간', '지원대상', '나이조건', '소득조건', '신청방법', '제출서류', '주관기관', '운영기관', '신청URL', '참고URL1', '참고URL2', '최초등록일시', '최종수정일시', '수집페이지', '수집출처']


## 1. 필수 컬럼 보정 및 기본 정리

In [2]:
required_cols = [
    "지역", "조회_zipCd", "정책ID", "정책명", "정책키워드", "정책설명", "정책지원내용",
    "정책대분류", "정책중분류", "자동분류", "대표분류", "신청기간", "사업기간",
    "지원대상", "나이조건", "소득조건", "신청방법", "제출서류", "주관기관", "운영기관",
    "신청URL", "참고URL1", "참고URL2", "최초등록일시", "최종수정일시", "수집페이지", "수집출처"
]
for col in required_cols:
    if col not in df_raw.columns:
        df_raw[col] = ""

df = df_raw.copy()
for col in required_cols:
    if col != "수집페이지":
        df[col] = df[col].fillna("").astype(str)

region_map = {
    "서울특별시": "서울", "경기도": "경기", "강원특별자치도": "강원", "강원도": "강원",
    "충청북도": "충북", "충청남도": "충남", "전북특별자치도": "전북", "전라북도": "전북",
    "전라남도": "전남", "경상북도": "경북", "경상남도": "경남", "제주특별자치도": "제주"
}
df["지역"] = df["지역"].str.strip().replace(region_map)
df["정책ID_정리"] = df["정책ID"].replace({"nan": "", "None": "", "NaN": ""}).fillna("").astype(str).str.strip()
df["정책명_정리"] = df["정책명"].fillna("").astype(str).str.strip()

print("지역:", sorted(df["지역"].dropna().unique()))
print("정리 후 크기:", df.shape)

지역: ['강원', '경기', '경남', '경북', '서울', '전남', '전북', '제주', '충남', '충북']
정리 후 크기: (5470, 29)


## 2. 텍스트 정제 및 분석 텍스트 생성

`나이조건`, `소득조건`은 `지원대상`에 중복 포함되는 경우가 많으므로 분석 텍스트에는 별도로 넣지 않습니다.

In [3]:
def clean_policy_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&[a-zA-Z]+;", " ", text)
    text = re.sub(r"연령제한\s*:\s*[YN]", " ", text)
    text = re.sub(r"\b\d{7}\b", " ", text)
    text = re.sub(r"\b\d{6}\b", " ", text)
    text = re.sub(r"\b0\s+0\b", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

analysis_text_cols = ["정책명", "정책키워드", "정책설명", "정책지원내용", "지원대상", "신청방법", "주관기관", "운영기관"]
for col in analysis_text_cols:
    df[f"{col}_정제"] = df[col].apply(clean_policy_text)

df["분석텍스트"] = df[[f"{col}_정제" for col in analysis_text_cols]].agg(" ".join, axis=1).apply(clean_policy_text)
df[["정책명", "정책키워드", "분석텍스트"]].head(3)

,정책명,정책키워드,분석텍스트
0,청년미래적금,보조금,"청년미래적금 보조금 청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월 최대 50만원 한도 내에서 자유롭게 납입 가능(2026년 6월 22일 출시 예정) 은행이자+비과세 혜택+정부기여금(납입금액에 비례해 일반형 ..."
1,(농식품부) 농식품 바우처,바우처,(농식품부) 농식품 바우처 바우처 『농업·농촌 및 식품산업 기본법』 제 23조의 2(취약계층 등에 대한 식품지원)에 근거하여 취약계층의 식품 접근성을 강화하고 균형 있는 식품 섭취를 지원하는 식품지원 제도 - 지원방식: 전자바우처(카드방식) -...
2,(문체부) 청년예술인 예술활동 적립계좌,보조금,(문체부) 청년예술인 예술활동 적립계좌 보조금 청년예술인에게 중장기 자산형성의 기회를 마련하여 안정적인 예술활동을 지원하는 사업 매월 일정 금액을 24개월간 적금 저축 시 가입자가 저축한 금액만큼 정부지원금을 지원 (1인 최대 240만원) - ...


## 3. 정책 재분류

공식 대분류·중분류와 정책 텍스트 키워드를 함께 활용해 `일자리`, `직무교육`, `주거지원`, `창업지원`, `복지`, `참여 프로그램`, `기타`로 재분류합니다.

In [4]:
CATEGORY_KEYWORDS = {
    "일자리": ["취업", "채용", "고용", "구직", "일자리", "면접", "인턴", "근로", "취업지원", "고용지원", "취업연계", "일경험"],
    "직무교육": ["교육", "훈련", "직무", "자격증", "역량", "인재양성", "실습", "강의", "과정", "취업교육", "디지털", "AI", "코딩"],
    "주거지원": ["주거", "월세", "전세", "임대", "주택", "보증금", "기숙사", "청년주택", "전월세", "주거비"],
    "창업지원": ["창업", "스타트업", "사업화", "예비창업", "창업자", "창업공간", "사업자", "소상공인", "기업가"],
    "복지": ["복지", "금융", "수당", "지원금", "생활비", "교통비", "식비", "건강", "심리", "상담", "문화", "자산", "적금", "바우처"],
    "참여 프로그램": ["참여", "프로그램", "동아리", "네트워크", "커뮤니티", "공모전", "멘토링", "청년활동", "위원회", "서포터즈", "교류"],
}
CATEGORY_PRIORITY = ["창업지원", "주거지원", "직무교육", "일자리", "복지", "참여 프로그램"]

def classify_policy_solo(row):
    official = f"{row.get('정책대분류', '')} {row.get('정책중분류', '')} {row.get('자동분류', '')} {row.get('대표분류', '')}"
    text = f"{official} {row.get('분석텍스트', '')}".lower()
    matched = []
    if "창업" in official:
        matched.append("창업지원")
    if "주거" in official:
        matched.append("주거지원")
    if any(k in official for k in ["교육", "역량", "훈련"]):
        matched.append("직무교육")
    if any(k in official for k in ["일자리", "취업", "고용"]):
        matched.append("일자리")
    if any(k in official for k in ["금융", "복지", "문화", "건강"]):
        matched.append("복지")
    if any(k in official for k in ["참여", "권리"]):
        matched.append("참여 프로그램")

    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw.lower() in text for kw in keywords):
            matched.append(category)

    matched_unique = []
    for m in matched:
        if m not in matched_unique:
            matched_unique.append(m)
    if not matched_unique:
        matched_unique = ["기타"]

    representative = "기타"
    for cat in CATEGORY_PRIORITY:
        if cat in matched_unique:
            representative = cat
            break
    if representative == "기타":
        representative = matched_unique[0]
    return pd.Series({"대표분류_개선": representative, "매칭분류_전체": ", ".join(matched_unique), "매칭분류수": len([x for x in matched_unique if x != "기타"])})

classified = df.apply(classify_policy_solo, axis=1)
df = pd.concat([df, classified], axis=1)

display(df["대표분류_개선"].value_counts().reset_index().rename(columns={"index": "분류", "대표분류_개선": "정책수"}))

,분류,정책수
0,직무교육,2481
1,창업지원,1473
2,일자리,539
3,주거지원,487
4,복지,440
5,참여 프로그램,50


## 4. 중복 제거

지역별 정책 공급량을 분석하므로 `지역 + 정책ID` 기준으로 중복을 제거합니다. 동일 정책ID가 여러 지역에 나타나는 것은 전국 공통 정책일 수 있어 제거하지 않습니다.

In [5]:
before = len(df)
df["_최종수정일시_dt"] = pd.to_datetime(df["최종수정일시"], errors="coerce")
df = df.sort_values(["지역", "_최종수정일시_dt"], ascending=[True, False])
has_id = df["정책ID_정리"].ne("")
df_with_id = df[has_id].drop_duplicates(subset=["지역", "정책ID_정리"], keep="first")
df_without_id = df[~has_id].drop_duplicates(subset=["지역", "정책명_정리"], keep="first")
df = pd.concat([df_with_id, df_without_id], ignore_index=True)
after = len(df)

df_diag = df_raw.copy()
df_diag["정책ID_정리"] = df_diag["정책ID"].fillna("").astype(str).str.strip()
df_diag["정책명_정리"] = df_diag["정책명"].fillna("").astype(str).str.strip()
duplicate_diagnostics = pd.DataFrame({
    "진단항목": ["전체 행 수", "지역+정책ID 중복 수", "정책ID만 기준 중복 수", "지역+정책명 중복 수", "정책명만 기준 중복 수", "전처리 후 행 수"],
    "값": [before, df_diag.duplicated(subset=["지역", "정책ID_정리"]).sum(), df_diag.duplicated(subset=["정책ID_정리"]).sum(), df_diag.duplicated(subset=["지역", "정책명_정리"]).sum(), df_diag.duplicated(subset=["정책명_정리"]).sum(), after]
})
print(f"중복 제거 전: {before:,}건")
print(f"중복 제거 후: {after:,}건")
print(f"제거된 중복: {before - after:,}건")
display(duplicate_diagnostics)

중복 제거 전: 5,470건
중복 제거 후: 5,470건
제거된 중복: 0건


,진단항목,값
0,전체 행 수,5470
1,지역+정책ID 중복 수,0
2,정책ID만 기준 중복 수,3665
3,지역+정책명 중복 수,221
4,정책명만 기준 중복 수,3836
5,전처리 후 행 수,5470


## 5. 날짜·신청 가능성·정보 접근성 변수 생성

In [6]:
def extract_dates(text):
    if pd.isna(text):
        return []
    text = str(text)
    patterns = [
        r"(20\d{2})(\d{2})(\d{2})",
        r"(20\d{2})[-./]\s*(\d{1,2})[-./]\s*(\d{1,2})",
        r"(20\d{2})년\s*(\d{1,2})월\s*(\d{1,2})일",
    ]
    dates = []
    for pat in patterns:
        for m in re.finditer(pat, text):
            y, mo, d = map(int, m.groups())
            try:
                dates.append(pd.Timestamp(year=y, month=mo, day=d))
            except ValueError:
                pass
    return dates

reference_date = pd.Timestamp.today().normalize()
period_text = df["신청기간"].fillna("") + " " + df["사업기간"].fillna("")
date_lists = period_text.apply(extract_dates)
df["시작일_추정"] = date_lists.apply(lambda xs: min(xs) if xs else pd.NaT)
df["종료일_추정"] = date_lists.apply(lambda xs: max(xs) if xs else pd.NaT)
df["상시성정책"] = period_text.str.contains("상시|연중|계속|수시|예산소진|별도 공고|별도공고", regex=True, na=False).astype(int)
df["신청가능_추정"] = (df["상시성정책"].eq(1) | df["종료일_추정"].isna() | (df["종료일_추정"] >= reference_date)).astype(int)
df["신청URL_존재"] = df["신청URL"].fillna("").astype(str).str.startswith("http").astype(int)
df["참고URL_존재"] = (df["참고URL1"].fillna("").astype(str).str.startswith("http") | df["참고URL2"].fillna("").astype(str).str.startswith("http")).astype(int)
df["신청방법_존재"] = df["신청방법"].fillna("").astype(str).str.strip().ne("").astype(int)
df["설명길이"] = df["분석텍스트"].str.len()
print("기준일:", reference_date.date())
display(df[["정책명", "신청기간", "사업기간", "시작일_추정", "종료일_추정", "상시성정책", "신청가능_추정"]].head())

기준일: 2026-06-12


,정책명,신청기간,사업기간,시작일_추정,종료일_추정,상시성정책,신청가능_추정
0,청년미래적금,20260622 ~ 20261231,20260622 ~ 20261231,2026-06-22,2026-12-31,0,1
1,스마트 모빌리티 창업캠프사업,20260401 ~ 20260528,20260301 ~ 20261231,2026-03-01,2026-12-31,0,1
2,산림산업 창업지원_청년 임팩트 창업 아이디어 챌린지,20260520 ~ 20260603,20260622 ~ 20260623,2026-05-20,2026-06-23,0,1
3,(농식품부) 농식품 바우처,20251222 ~ 20261211,20260102 ~ 20261231,2025-12-22,2026-12-31,0,1
4,삼척형 청년인턴 지원사업,20260202 ~ 20260206,20260223 ~ 20260628,2026-02-02,2026-06-28,0,1


## 6. 지역·분류 요약 및 저장

In [7]:
region_summary = df.groupby("지역").agg(
    정책수=("정책ID_정리", "count"),
    고유정책수=("정책ID_정리", "nunique"),
    신청가능정책수=("신청가능_추정", "sum"),
    신청URL보유정책수=("신청URL_존재", "sum"),
    신청방법보유정책수=("신청방법_존재", "sum"),
    평균설명길이=("설명길이", "mean"),
    평균매칭분류수=("매칭분류수", "mean"),
).reset_index()
region_summary["신청가능비율"] = (region_summary["신청가능정책수"] / region_summary["정책수"]).round(4)
region_summary["신청URL보유비율"] = (region_summary["신청URL보유정책수"] / region_summary["정책수"]).round(4)
region_summary["신청방법보유비율"] = (region_summary["신청방법보유정책수"] / region_summary["정책수"]).round(4)

category_summary = df.groupby("대표분류_개선").agg(
    정책수=("정책ID_정리", "count"),
    지역수=("지역", "nunique"),
    신청가능정책수=("신청가능_추정", "sum"),
    평균설명길이=("설명길이", "mean"),
).reset_index().sort_values("정책수", ascending=False)

region_category_pivot = pd.pivot_table(df, index="지역", columns="대표분류_개선", values="정책ID_정리", aggfunc="count", fill_value=0).reset_index()

display(region_summary)
display(category_summary)
display(region_category_pivot)

df.to_csv(OUTPUT_DIR / "policy_preprocessed_solo.csv", index=False, encoding="utf-8-sig")
region_summary.to_csv(OUTPUT_DIR / "policy_region_summary_solo.csv", index=False, encoding="utf-8-sig")
category_summary.to_csv(OUTPUT_DIR / "policy_category_summary_solo.csv", index=False, encoding="utf-8-sig")
region_category_pivot.to_csv(OUTPUT_DIR / "policy_region_category_pivot_solo.csv", index=False, encoding="utf-8-sig")
duplicate_diagnostics.to_csv(OUTPUT_DIR / "policy_duplicate_diagnostics_solo.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", OUTPUT_DIR)

,지역,정책수,고유정책수,신청가능정책수,신청URL보유정책수,신청방법보유정책수,평균설명길이,평균매칭분류수,신청가능비율,신청URL보유비율,신청방법보유비율
0,강원,489,489,255,207,325,401.378323,2.789366,0.5215,0.4233,0.6646
1,경기,515,515,298,222,326,387.095146,2.735922,0.5786,0.4311,0.6330
2,경남,584,584,278,204,315,386.662671,2.816781,0.4760,0.3493,0.5394
3,경북,539,539,307,209,318,408.480519,2.821892,0.5696,0.3878,0.5900
4,서울,461,461,243,201,281,444.260304,2.830803,0.5271,0.4360,0.6095
5,전남,535,535,297,194,324,387.151402,2.728972,0.5551,0.3626,0.6056
6,전북,534,534,306,191,317,396.644195,2.677903,0.5730,0.3577,0.5936
7,제주,608,608,341,296,401,395.777961,2.768092,0.5609,0.4868,0.6595
8,충남,707,707,468,236,434,317.212164,2.633663,0.6620,0.3338,0.6139
9,충북,498,498,253,214,335,403.399598,2.783133,0.5080,0.4297,0.6727


,대표분류_개선,정책수,지역수,신청가능정책수,평균설명길이
3,직무교육,2481,10,1341,384.771866
5,창업지원,1473,10,787,352.604209
1,일자리,539,10,297,473.500928
2,주거지원,487,10,309,466.540041
0,복지,440,10,279,357.500000
4,참여 프로그램,50,10,33,348.320000


대표분류_개선,지역,복지,일자리,주거지원,직무교육,참여 프로그램,창업지원
0,강원,37,47,32,230,5,138
1,경기,43,53,43,244,5,127
2,경남,46,53,64,259,3,159
3,경북,35,56,49,226,5,168
4,서울,34,44,40,216,3,124
5,전남,42,49,45,241,4,154
6,전북,49,62,45,232,4,142
7,제주,47,61,58,277,5,160
8,충남,66,61,64,331,13,172
9,충북,41,53,47,225,3,129


저장 완료: C:\Users\yong\Desktop\TM\tm2\textmining_solo_policy_project\textmining_solo_policy_project\outputs


## 보고서용 전처리 기준 요약

본 분석에서는 온통청년 API로 수집한 청년정책 데이터를 지역별 정책 공급 현황과 정책 키워드 대응도를 분석하기 위한 기초 데이터로 사용하였다. 결측 텍스트는 빈 문자열로 대체하였고, 정책명·정책키워드·정책설명·정책지원내용·지원대상·신청방법·주관기관·운영기관을 결합하여 텍스트마이닝용 분석 텍스트를 구성하였다. API 응답에 포함된 코드성 숫자, HTML 태그, 반복 공백, 연령제한 코드 등은 의미 없는 토큰으로 판단하여 제거하였다. 중복 제거는 지역별 정책 공급량 분석 목적에 맞추어 `지역 + 정책ID` 기준으로 수행하였다. 동일 정책ID가 여러 지역에 나타나는 경우는 전국 공통 정책 또는 복수 지역 대상 정책일 수 있으므로 제거하지 않았다.